In [1]:

import pandas as pd
import numpy as np

#NETTOYAGE
import re
import string
import nltk
"""nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')"""
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

#FEATURES
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

#BERT
# 1. On charge TensorFlow
import tensorflow as tf

# 2. On charge les outils Keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GlobalAveragePooling1D

# 3. On charge Transformers (Maintenant il va détecter TF grâce à tf-keras)
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification

# --- SECTION TRACKING ---
import mlflow
import mlflow.sklearn
import mlflow.tensorflow

# Importation du jeu

In [2]:
columns_name = ['TARGET', 'id', 'date', '??', 'user', 'tweet']

df = pd.read_csv("../Data/training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', names=columns_name)

print(df["tweet"][df["tweet"].isnull()])

print(df.head())
print(df["tweet"][df["tweet"] = 1])

Series([], Name: tweet, dtype: object)
   TARGET          id                          date        ??  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                              tweet  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  
         TARGET          id                          date        ??  \
0             0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY

In [15]:
print(df["tweet"][800000])

I LOVE @Health4UandPets u guys r the best!! 


# Nettoyage du texte

## Fonction

In [3]:
#Fonction pour nettoyer le texte : lemmatisation
def nettoyer_texte_lem(texte):
    if not isinstance(texte, str):
        return ""

    #Passer en minuscule tout le texte
    texte = texte.lower()

    #Supprimer des éléments frequents dans des tweets, mais jugés ininteressants pour l'analyse
    texte = re.sub(r'<.*?>', '', texte)
    texte = re.sub(r'@\w+', '', texte)
    texte = re.sub(r'\d+', '', texte)
    texte = re.sub(r'https?://\S+|www\.\S+', '', texte)

    #Supprimer les ponctuactions
    texte = texte.translate(str.maketrans("","",string.punctuation))

    texte = re.sub(r'\s+', ' ', texte)

    #Initialisation
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    #TOKENISATION
    tokens = texte.split()

    #Parcours des tokens:
    clean_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    #Renvoie des elements à joindre
    return " ".join(clean_tokens)


#Fonction pour nettoyer le texte : stemming
def nettoyer_texte_stem(texte):
    if not isinstance(texte, str):
        return ""
    
    texte = texte.lower()
    texte = re.sub(r'<.*?>', '', texte)
    texte = re.sub(r'@\w+', '', texte)
    texte = re.sub(r'\d+', '', texte)
    texte = re.sub(r'https?://\S+|www\.\S+', '', texte)
    
    #Supprimer les ponctuactions
    texte = texte.translate(str.maketrans("", "", string.punctuation))
    
    texte = re.sub(r'\s+', ' ', texte).strip()

    #Initialisation
    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()

    tokens = texte.split()
    clean_tokens = [
        stemmer.stem(token)          # <-- stem au lieu de lemmatize
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    return " ".join(clean_tokens)

# Comaraison entre tweet de base, et les deux methodes de nettoyage
exemple = df['tweet'][0]
print("Original :", exemple)
print("Lemmatisation :", nettoyer_texte(exemple))
print("Stemming :", nettoyer_texte_stemming(exemple))

## Application

In [4]:
df_trt = df.copy()[['TARGET', 'id', 'date','user', 'tweet']]

df_trt['tweet_net'] = df_trt['tweet'].apply(nettoyer_texte_le)
print(df_trt['tweet'], df_trt['tweet_net'])

0          @switchfoot http://twitpic.com/2y1zl - Awww, t...
1          is upset that he can't update his Facebook by ...
2          @Kenichan I dived many times for the ball. Man...
3            my whole body feels itchy and like its on fire 
4          @nationwideclass no, it's not behaving at all....
                                 ...                        
1599995    Just woke up. Having no school is the best fee...
1599996    TheWDB.com - Very cool to hear old Walt interv...
1599997    Are you ready for your MoJo Makeover? Ask me f...
1599998    Happy 38th Birthday to my boo of alll time!!! ...
1599999    happy #charitytuesday @theNSPCC @SparksCharity...
Name: tweet, Length: 1600000, dtype: object 0          awww thats bummer shoulda got david carr third...
1          upset cant update facebook texting might cry r...
2               dived many time ball managed save rest bound
3                            whole body feel itchy like fire
4                                      be

# Featuring

## pre entrainement

In [5]:
#Definition des variables
X = df_trt['tweet_net']
y = df_trt['TARGET']
y = y.replace(4, 1)

#Separation : 80% train / 20% reste
X_train, X_reste, y_train, y_reste = train_test_split(X, y, test_size=0.2, random_state=42)
#Separation du reste : 50% validation / 50% test
X_val, X_test, y_val, y_test = train_test_split(X_reste, y_reste, test_size=0.5, random_state=42)

## ML FLOW

### Modèles simples

In [6]:
mlflow.set_experiment("Test_modeles")

modeles = [
    {
        "nom": "Logistic_Regression",
        "modele": LogisticRegression(max_iter=1000, C=1.0),
        "params": {"max_iter": 1000, "C": 1.0}
    },
    {
        "nom": "Random_Forest",
        "modele": RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1),
        "params": {"n_estimators": 50, "max_depth": 10}
    }
]

for config in modeles:
    with mlflow.start_run(run_name=config['nom']):

        print(f"--- Entraînement de : {config['nom']} ---")

        # 1. Création de la Pipeline (Transformation + Modèle)
        # TfidfVectorizer : max_features=5000 pour limiter la mémoire
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=5000)),
            ('clf', config['modele'])
        ])

        # 2. Entraînement
        pipe.fit(X_train, y_train)

        # 3. Prédiction
        y_pred = pipe.predict(X_test)

        # 4. Calcul des métriques
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy : {acc:.4f}")

        # 5. LOGGING MLFLOW
        # On enregistre les paramètres donnés au début
        mlflow.log_params(config['params'])

        # On enregistre la performance
        mlflow.log_metric("accuracy", acc)

        # On enregistre le modèle COMPLET (Pipeline)
        # C'est ce fichier qu'on chargera dans l'API Azure plus tard
        mlflow.sklearn.log_model(pipe, "model")

2026/01/30 13:20:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/01/30 13:20:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/01/30 13:20:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/01/30 13:20:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/01/30 13:20:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/01/30 13:20:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/01/30 13:20:25 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/30 13:20:25 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/30 13:20:25 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Cmd('git') failed due to: exit code(69)
  cmdline: git version
  stderr: 'You have not agreed to th

--- Entraînement de : Logistic_Regression ---


2026/01/30 13:20:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy : 0.7714


/opt/anaconda3/envs/air_paradis_final/lib/python3.10/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


--- Entraînement de : Random_Forest ---


2026/01/30 13:20:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy : 0.6992


/opt/anaconda3/envs/air_paradis_final/lib/python3.10/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


### Modèles plus avancées

In [ ]:
# --- 1. PRÉPARATION DES DONNÉES ---
MAX_WORDS = 10000       # On garde les 10000 mots les plus fréquents
MAX_LEN = 100           # On limite les tweets à 100 mots

# A. Initialisation du Tokenizer
print("Token")
tokenizer = Tokenizer(num_words=MAX_WORDS)
# IMPORTANT : On n'apprend le vocabulaire QUE sur le train set pour éviter la fuite de données
tokenizer.fit_on_texts(X_train[:2000])

# B. Conversion Texte -> Séquence de nombres
X_train_seq = tokenizer.texts_to_sequences(X_train[:2000])
X_val_seq = tokenizer.texts_to_sequences(X_val[:2000])
X_test_seq = tokenizer.texts_to_sequences(X_test[:2000])

# C. Padding (Remplissage pour avoir la même taille partout)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

#### KERAS CLASSIQUE

In [ ]:
#Entrainement
mlflow.set_experiment("Test_modeles")
mlflow.tensorflow.autolog()

print("RUN")
with mlflow.start_run(run_name="Keras_Embedding_Simple"):

    # Construction du modèle
    model_embed = Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
        GlobalAveragePooling1D(),        # Moyenne des embeddings (pas de mémoire séquentielle)
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')  # Sortie binaire : positif / négatif
    ])

    model_embed.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model_embed.summary()

    print("Entraînement KERAS en cours...")
    history_embed = model_embed.fit(
        X_train_pad, y_train[:2000],
        epochs=5,
        batch_size=32,
        validation_data=(X_val_pad, y_val[:2000]),
        verbose=1
    )

    # Évaluation
    loss, acc = model_embed.evaluate(X_test_pad, y_test[:2000], verbose=0)

    # Log MLflow
    mlflow.log_param("model_type", "Keras_Embedding_Simple")
    mlflow.log_param("embedding_dim", 64)
    mlflow.log_param("epochs", 5)
    mlflow.log_param("batch_size", 32)
    mlflow.log_metric("accuracy", acc)
    
    print("Sauvegarde du modèle KERAS vers MLflow...")
    
    # C'est ici que ça change par rapport à Sklearn
    mlflow.tensorflow.log_model(
        model, 
        "model_KERAS", # Nom du dossier dans MLflow
        keras_model_kwargs={"save_format": "h5"} # Force le format .h5 (plus compatible)
    )

    print("KERAS terminé et tracké sur MLflow.")

    print(f"\n✅ Modèle basique terminé — Accuracy: {acc:.4f}")

#### LSTM

#### sans embedding

In [9]:
#Entrainement
mlflow.set_experiment("Test_modeles")
mlflow.tensorflow.autolog()

print("RUN")
with mlflow.start_run(run_name="LSTM_Keras"):

    # Construction du modèle
    model = Sequential()
    # Couche Embedding : transforme les entiers en vecteurs denses
    model.add(Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN))
    # Couche LSTM
    model.add(LSTM(64, dropout=0.2))
    # Couche de sortie
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    print("Entraînement LSTM en cours...")
    model.fit(X_train_pad, y_train[:2000],
              validation_data=(X_val_pad, y_val[:2000]),
              epochs=3,
              batch_size=64)
    
    print("Sauvegarde du modèle LSTM vers MLflow...")
    
    # C'est ici que ça change par rapport à Sklearn
    mlflow.tensorflow.log_model(
        model, 
        "model_2_lstm", # Nom du dossier dans MLflow
        keras_model_kwargs={"save_format": "h5"} # Force le format .h5 (plus compatible)
    )

    print("LSTM terminé et tracké sur MLflow.")

Token


2026/01/30 13:24:39 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.16.0 <= tensorflow, but the installed version is 2.15.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


RUN


2026/01/30 13:24:39 WARNING mlflow.tensorflow: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


Entraînement LSTM en cours...
Epoch 1/3
32/32 [==============================] - 3s 68ms/step - loss: 0.6905 - accuracy: 0.5565 - val_loss: 0.6842 - val_accuracy: 0.6080
Epoch 2/3
 3/32 [=>............................] - ETA: 1s - loss: 0.6741 - accuracy: 0.7188

...

INFO:tensorflow:Assets written to: /var/folders/gq/31hn8tsj6yngx1gn3qmw7fc40000gn/T/tmpgmacmlf1/model/data/model/assets


INFO:tensorflow:Assets written to: /var/folders/gq/31hn8tsj6yngx1gn3qmw7fc40000gn/T/tmpgmacmlf1/model/data/model/assets
2026/01/30 13:24:53 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /var/folders/gq/31hn8tsj6yngx1gn3qmw7fc40000gn/T/tmpgmacmlf1/model, flavor: tensorflow). Fall back to return ['tensorflow==2.15.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/01/30 13:24:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/30 13:24:54 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


Sauvegarde du modèle LSTM vers MLflow...


/opt/anaconda3/envs/air_paradis_final/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


LSTM terminé et tracké sur MLflow.


#### avec embedding (Word2Vec)

In [ ]:
EMBEDDING_DIM = 100

X_train_tokens = [text.split() for text in X_train[:50000]]

# Entraînement
print("Entraînement de Word2Vec...")
w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=EMBEDDING_DIM,  
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

In [ ]:
word_index = tokenizer.word_index
num_words = min(MAX_WORDS, len(word_index) + 1)

# Création de la matrice : chaque ligne = le vecteur Word2Vec du mot
embedding_matrix_w2v = np.zeros((num_words, EMBEDDING_DIM))
mots_trouves = 0

for word, i in word_index.items():
    if i >= MAX_WORDS:
        continue
    if word in w2v_model.wv:
        embedding_matrix_w2v[i] = w2v_model.wv[word]
        mots_trouves += 1

In [ ]:
mlflow.set_experiment("Test_modeles")

with mlflow.start_run(run_name="LSTM_Word2Vec"):

    model_w2v = Sequential([
        Embedding(
            input_dim=num_words,
            output_dim=EMBEDDING_DIM,
            weights=[embedding_matrix_w2v],   # <-- Poids Word2Vec
            input_length=MAX_LEN,
            trainable=False                    # On gèle les poids
        ),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model_w2v.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model_w2v.summary()

    print("Entraînement LSTM + Word2Vec en cours...")
    history_w2v = model_w2v.fit(
        X_train_pad, y_train[:2000],
        epochs=5,
        batch_size=32,
        validation_data=(X_val_pad, y_val[:2000]),
        verbose=1
    )

    # Évaluation
    loss, acc = model_w2v.evaluate(X_test_pad, y_test[:2000], verbose=0)

    # Log MLflow
    mlflow.log_param("model_type", "LSTM_Word2Vec")
    mlflow.log_param("embedding_dim", EMBEDDING_DIM)
    mlflow.log_param("embedding_type", "Word2Vec_custom")
    mlflow.log_param("trainable_embedding", False)
    mlflow.log_param("w2v_window", 5)
    mlflow.log_param("w2v_min_count", 2)
    mlflow.log_param("w2v_epochs", 10)
    mlflow.log_param("lstm_units", 64)
    mlflow.log_param("epochs", 5)
    mlflow.log_param("batch_size", 32)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("loss", loss)

    mlflow.tensorflow.log_model(
        model_w2v,
        "model_LSTM_Word2Vec",
        keras_model_kwargs={"save_format": "h5"}
    )

#### BERT

In [7]:
#TOKENISER de DistilBERT
bert_token = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


def bert_encode(texts, max_len=100):
    return bert_token(
        texts.tolist(),
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors='tf'
    )
#ECHANTILLONNAGE : réduction du jeu
print("ECHANTILLONNAGE : réduction du jeu")
train_encode = bert_encode(X_train[:2000])
val_encode = bert_encode(X_val[:2000])
test_encode = bert_encode(X_test[:2000])

#Transformation en Dataset
print("Transformation en Dataset")
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encode), y_train[:2000])).shuffle(1000).batch(16)
val_dataset = tf.data.Dataset.from_tensor_slices((dict(val_encode), y_val[:2000])).batch(16)
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encode), y_test[:2000])).batch(16)


/opt/anaconda3/envs/air_paradis_final/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


ECHANTILLONNAGE : réduction du jeu
Transformation en Dataset


2026-01-30 13:20:48.007337: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-01-30 13:20:48.007400: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-30 13:20:48.007412: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-30 13:20:48.007604: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-30 13:20:48.007964: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [8]:
mlflow.set_experiment("Test_modeles")
mlflow.tensorflow.autolog()

print("RUN")
with mlflow.start_run(run_name="DistilBERT_FineTuning"):
    model_bert = TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=1)

    # On utilise un learning rate TRES FAIBLE (5e-5) pour ne pas "casser" le cerveau pré-entraîné
    optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)

    # Compilation (La loss est souvent gérée en interne par HuggingFace, mais on précise pour Keras)
    print("Compilation")
    loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)
    
    model_bert.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

    # 2 epochs suffisent généralement pour BERT
    print("Execution")
    model_bert.fit(train_dataset, validation_data=val_dataset, epochs=2)

    print("Sauvegarde du modèle BERT vers MLflow...")
    
    # C'est ici que ça change par rapport à Sklearn
    mlflow.tensorflow.log_model(
        model_bert, 
        "model_2_bert", # Nom du dossier dans MLflow
        keras_model_kwargs={"save_format": "tf"} # Force le format .h5 (plus compatible)
    )
    
    print("Fin")

2026/01/30 13:20:48 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.16.0 <= tensorflow, but the installed version is 2.15.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


RUN


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Compilation
Execution
Epoch 1/2


2026-01-30 13:20:54.144108: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-01-30 13:20:54.591019: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp_10.


...

INFO:tensorflow:Assets written to: /var/folders/gq/31hn8tsj6yngx1gn3qmw7fc40000gn/T/tmpyhe5r2sl/model/data/model/assets
2026/01/30 13:24:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/30 13:24:23 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


Sauvegarde du modèle BERT vers MLflow...


INFO:tensorflow:Assets written to: /var/folders/gq/31hn8tsj6yngx1gn3qmw7fc40000gn/T/tmpxqcsmnvb/model/data/model/assets


INFO:tensorflow:Assets written to: /var/folders/gq/31hn8tsj6yngx1gn3qmw7fc40000gn/T/tmpxqcsmnvb/model/data/model/assets


Fin
